In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import plotly.graph_objects as bg
from plotly.subplots import make_subplots

In [ ]:
df= pd.read_csv('condo_month.csv')
id_cols = ["RegionID", "SizeRank", "RegionName", "RegionType", "StateName"]

df = df.melt(id_vars=id_cols, var_name='Date', value_name='Value')

df_long = df.copy()
df_long.head()

In [ ]:
df[(df['RegionName'] == 'United States') & (df_long['StateName'].isna())]

In [ ]:
df['Date'] = pd.to_datetime(df['Date'])

# df.loc[df['RegionName'] == 'United States', 'StateName'] = df['StateName'].fillna('USA')

df['StateName'] = np.where(df['RegionName'] == 'United States', 'USA', df['StateName'])
df.head()

In [ ]:
df = df.dropna(subset=['Value'])

In [ ]:
df = df.sort_values(by=['RegionID', 'Date']).reset_index(drop=True)
df.head()

In [ ]:
df_long = df.copy()
# Month-over-Month (MoM) & Year-over-Year (YoY) Appreciation
df_long['MoM_Growth'] = df_long.groupby('RegionID')['Value'].pct_change(periods=1) * 100
df_long['YoY_Growth'] = df_long.groupby('RegionID')['Value'].pct_change(periods=12) * 100
df_long.tail()

In [ ]:
# Running Peak Calculations
df_long['Running_Peak'] = df_long.groupby('RegionID')['Value'].cummax()

# Drawdown Calculations
df_long['Drawdown_Pct'] = ((df_long['Value'] - df_long['Running_Peak']) / df_long['Running_Peak']) * 100

# Binary flag: True if market is currently at or above its previous high
df_long['Is_At_Peak'] = df_long['Value'] >= df_long['Running_Peak']

df_long.head()

In [ ]:
top_regions = df_long[df_long['RegionType'] == 'msa'].groupby('RegionName')['SizeRank'].min().nsmallest(5).index
df_sub = df_long[df_long['RegionName'].isin(top_regions)].copy()

In [ ]:
fig2 = px.line(
    df_sub, 
    x='Date', 
    y='YoY_Growth', 
    color='RegionName',
    title="Year-over-Year (YoY) Appreciation Rate (%)",
    labels={'YoY_Growth': 'YoY Growth (%)', 'Date': 'Year'}
)

# Add baseline zero reference line
fig2.add_hline(y=0, line_dash="dash", line_color="red", annotation_text="Zero Growth")
fig2.update_layout(height=450, hovermode="x unified")
fig2.show()

In [ ]:
fig1 = make_subplots(
    rows=2, cols=1, 
    shared_xaxes=True, 
    vertical_spacing=0.08,
    subplot_titles=("Home Value Trajectory", "Drawdown from All-Time High (%)")
)

for region in top_regions:
    region_data = df_sub[df_sub['RegionName'] == region]
    
    # Upper panel: Values
    fig1.add_trace(
        bg.Scatter(x=region_data['Date'], y=region_data['Value'], name=region, mode='lines'),
        row=1, col=1
    )
    
    # Lower panel: Drawdown %
    fig1.add_trace(
        bg.Scatter(x=region_data['Date'], y=region_data['Drawdown_Pct'], name=region, mode='lines', showlegend=False),
        row=2, col=1
    )

fig1.update_layout(height=650, title_text="Housing Market Peak-to-Trough Analysis", hovermode="x unified")
fig1.update_yaxes(title_text="Value ($)", row=1, col=1)
fig1.update_yaxes(title_text="Drawdown %", row=2, col=1)
fig1.show()

In [ ]:
fig2 = px.line(
    df_sub, 
    x='Date', 
    y='YoY_Growth', 
    color='RegionName',
    title="Year-Over-Year (YOY) Appreciation Rate (%)",
    labels={'YOY_Growth': 'YOY Growth (%)', 'Date': 'Year'}
)

# Add baseline zero reference line
fig2.add_hline(y=0, line_dash="dash", line_color="red", annotation_text="Zero Growth")
fig2.update_layout(height=450, hovermode="x unified")

In [ ]:
# Filter for recovery periods (where market is below historical peak)
df_recovery = df_sub[~df_sub['Is_At_Peak']].dropna(subset=['MoM_Growth'])

fig3 = px.box(
    df_recovery, 
    x='RegionName', 
    y='MoM_Growth', 
    color='RegionName',
    title="Monthly Recovery Velocity During Post-Peak Rebounds",
    labels={'MoM_Growth': 'Monthly Growth Rate (%) during Recovery', 'RegionName': 'Region'}
)

fig3.update_layout(height=450, showlegend=False)
fig3.show()

In [ ]:
# Create dedicated DataFrames for each geographical level
df_national = df_long[df_long['RegionType'] == 'country']  # National aggregate
df_metros   = df_long[df_long['RegionType'] == 'msa']      # Metro areas

In [ ]:
# Set renderer to open charts in your web browser
# pio.renderers.default = "browser"

# Filter for top 100 regions to keep the dropdown lightweight
selected_regions = df_metros['RegionName'].unique()[:1000]
df_filtered = df_metros[df_metros['RegionName'].isin(selected_regions)]

fig = go.Figure()

# Add a trace for each region (hidden by default except the first)
for i, region in enumerate(selected_regions):
    region_data = df_filtered[df_filtered['RegionName'] == region]
    fig.add_trace(
        go.Scatter(
            x=region_data['Date'],
            y=region_data['Value'],
            name=region,
            visible=(i == 0)  # Only first region is visible on load
        )
    )

# Build dropdown menu buttons
buttons = []
for i, region in enumerate(selected_regions):
    # Boolean mask: True only for the trace matching the selected index
    visibility = [False] * len(selected_regions)
    visibility[i] = True
    
    buttons.append(
        dict(
            label=region,
            method="update",
            args=[{"visible": visibility}, {"title": f"Home Value Trend: {region}"}]
        )
    )

# Attach dropdown to layout
fig.update_layout(
    updatemenus=[dict(active=0, buttons=buttons, x=0.1, y=1.15)],
    title=f"Home Value Trend: {selected_regions[0]}",
    xaxis_title="Date",
    yaxis_title="Home Value ($)",
    height=500,
)

fig.show()

In [ ]:
# State-level aggregation

df_state = df[df_long['StateName'] != 'USA'].sort_values('Date').groupby(
    ['StateName', 'Date'], as_index=False)['Value'].sum().sort_values(['StateName', 'Date']).reset_index(drop=True)  # State-level data


In [ ]:
df_state['Running_Peak'] = df_state.groupby('StateName')['Value'].cummax()
df_state['Drawdown_Pct'] = (df_state['Value'] - df_state['Running_Peak']) / df_state['Running_Peak']
df_state['Is_At_Peak'] = df_state['Value'] == df_state['Running_Peak']

df_state['MoM_Growth'] = df_state.groupby('StateName')['Value'].pct_change(1)
df_state['YoY_Growth'] = df_state.groupby('StateName')['Value'].pct_change(12)

df_state.head()

In [ ]:
fig = px.line(
    df_state, 
    x='Date', 
    y='Value', 
    color='StateName', 
    title='State-Level Home Value Trends'
)

# Hide all lines except the first state initially
for i, trace in enumerate(fig.data):
    if i != 0:
        trace.visible = False

# Build dropdown buttons for each state
buttons = []
for i, trace in enumerate(fig.data):
    # Boolean mask showing only the selected state's line
    visibility = [False] * len(fig.data)
    visibility[i] = True
    
    buttons.append(dict(
        label=trace.name,
        method="update",
        args=[{"visible": visibility}, {"title": f"Home Value Trend by state: {trace.name}"}]
    ))

# Add dropdown to chart layout
fig.update_layout(
    updatemenus=[dict(
        active=0,
        buttons=buttons,
        direction="down",
        showactive=True,
        x=0.1,
        y=1.15
    )],
    height=550,
)

fig.show()

In [ ]:
# Select top 5 states with highest latest home values
latest_date = df_state['Date'].max()
top_states = (
    df_state[df_state['Date'] == latest_date]
    .sort_values(by='Value', ascending=False)
    .head(5)['StateName']
    .tolist()
)

df_top = df_state[df_state['StateName'].isin(top_states)]

fig = px.line(
    df_top, 
    x='Date', 
    y='Value', 
    color='StateName', 
    title='Top 5 States by Home Value'
)
fig.show()

In [ ]:
CENSUS_REGIONS = {
    # Northeast
    'CT': 'Northeast', 'ME': 'Northeast', 'MA': 'Northeast', 'NH': 'Northeast',
    'RI': 'Northeast', 'VT': 'Northeast', 'NJ': 'Northeast', 'NY': 'Northeast', 'PA': 'Northeast',
    # Midwest
    'IL': 'Midwest', 'IN': 'Midwest', 'IA': 'Midwest', 'KS': 'Midwest', 'MI': 'Midwest',
    'MN': 'Midwest', 'MO': 'Midwest', 'NE': 'Midwest', 'ND': 'Midwest', 'OH': 'Midwest',
    'SD': 'Midwest', 'WI': 'Midwest',
    # South
    'AL': 'South', 'AR': 'South', 'DE': 'South', 'FL': 'South', 'GA': 'South',
    'KY': 'South', 'LA': 'South', 'MD': 'South', 'MS': 'South', 'NC': 'South',
    'OK': 'South', 'SC': 'South', 'TN': 'South', 'TX': 'South', 'VA': 'South',
    'WV': 'South', 'DC': 'South',
    # West
    'AK': 'West', 'AZ': 'West', 'CA': 'West', 'CO': 'West', 'HI': 'West',
    'ID': 'West', 'MT': 'West', 'NV': 'West', 'NM': 'West', 'OR': 'West',
    'UT': 'West', 'WA': 'West', 'WY': 'West'
}

# Map Census Region to your state DataFrame
df_state['Region'] = df_state['StateName'].map(CENSUS_REGIONS)

# Aggregate raw values by Region and Date (ignoring unmapped rows like 'USA')
df_region = (
    df_state.dropna(subset=['Region'])
    .groupby(['Region', 'Date'], as_index=False)['Value']
    .mean() # Average home value across states in that region
    .sort_values(['Region', 'Date'])
    .reset_index(drop=True)
)

# calculate metrics properly at the Regional aggregate level
df_region['YoY_Growth'] = df_region.groupby('Region')['Value'].pct_change(12) * 100
df_region['Running_Peak'] = df_region.groupby('Region')['Value'].cummax()
df_region['Drawdown_Pct'] = ((df_region['Value'] - df_region['Running_Peak']) / df_region['Running_Peak']) * 100

df_region.head()

In [ ]:
fig_val = px.line(
    df_region,
    x='Date',
    y='Value',
    color='Region',
    title='US Housing Market Trends by Census Region',
    labels={'Value': 'Average Home Value ($)', 'Date': 'Year'}
)
fig_val.update_layout(height=500, hovermode="x unified")
fig_val.show()